In [7]:
import pandas as pd
import pint
import pint_xarray
import xarray as xr

from imagematerials.constants import _IMAGE_REGIONS as image_regions

In [2]:
ureg = pint.UnitRegistry()
ureg.define('terakilometer = 1e12 * kilometer')
ureg.define('milkilometer = 1e6 * kilometer')

pkmGlobal_original = pd.read_csv("image_3_4_data/pkmGlobalTot.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)
tkmGlobal_original = pd.read_csv("image_3_4_data/tkmGlobalTot.csv", delimiter=";", usecols=range(8),skiprows=3).drop(0)
load_original = pd.read_csv("image_3_4_data/load.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)

In [3]:
def convert_to_xarray(df, value_name, variable):
    df = df.rename(columns={"Unnamed: 0":"year","Unnamed: 1":"region"})
    df = (
        df.astype({"year": int, "region": int})  # Convert to integers
        .query("region not in [27, 28] and year in [2019, 2060]")  # Filter regions and years
    )
    # Convert all other columns (modes) to float
    df.iloc[:, 2:] = df.iloc[:, 2:].astype(float)

    # Melt the DataFrame so that transport modes become a single coordinate
    df = df.melt(id_vars=["year", "region"], var_name= variable, value_name=value_name)

    # Convert to xarray DataArray
    dataarray = df.set_index(["region", "year", "mode"]).to_xarray()[value_name]
    return dataarray

In [4]:
pkmGlobal = convert_to_xarray(pkmGlobal_original, value_name = "pkm", variable = "mode")
tkmGlobal = convert_to_xarray(tkmGlobal_original, value_name = "tkm", variable = "mode")
load = convert_to_xarray(load_original, value_name = "load", variable = "mode")


In [5]:
# Convert units
pkmGlobal = pkmGlobal.pint.quantify('terakilometer') 
pkmGlobal= pkmGlobal.pint.to("km")

In [10]:
image_regions


['CAN',
 'USA',
 'MEX',
 'RCAM',
 'BRA',
 'RSAM',
 'NAF',
 'WAF',
 'EAF',
 'SAF',
 'WEU',
 'CEU',
 'TUR',
 'UKR',
 'STAN',
 'RUS',
 'ME',
 'INDIA',
 'KOR',
 'CHN',
 'SEAS',
 'INDO',
 'JAP',
 'OCE',
 'RSAS',
 'RSAF']